# ML Assignment
- Name: Lazarus Abraham
- BITS ID: 2026AJ05230

## Import Modules

In [3]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_score,
    recall_score, f1_score, matthews_corrcoef
)
import os

## Data Cleaning and Processing

In [5]:
df=pd.read_csv("heart.csv")

In [6]:
df

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,52,1,0,125,212,0,1,168,0,1.0,2,2,3,0
1,53,1,0,140,203,1,0,155,1,3.1,0,0,3,0
2,70,1,0,145,174,0,1,125,1,2.6,0,0,3,0
3,61,1,0,148,203,0,1,161,0,0.0,2,1,3,0
4,62,0,0,138,294,1,1,106,0,1.9,1,3,2,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1020,59,1,1,140,221,0,1,164,1,0.0,2,0,2,1
1021,60,1,0,125,258,0,0,141,1,2.8,1,1,3,0
1022,47,1,0,110,275,0,0,118,1,1.0,1,1,2,0
1023,50,0,0,110,254,0,0,159,0,0.0,2,0,2,1


In [7]:
df["target"].value_counts()

target
1    526
0    499
Name: count, dtype: int64

In [8]:
df.isna().count()

age         1025
sex         1025
cp          1025
trestbps    1025
chol        1025
fbs         1025
restecg     1025
thalach     1025
exang       1025
oldpeak     1025
slope       1025
ca          1025
thal        1025
target      1025
dtype: int64

In [9]:
y = df["target"]
x = df.drop(columns=["target"])

In [10]:
y

0       0
1       0
2       0
3       0
4       0
       ..
1020    1
1021    0
1022    0
1023    1
1024    0
Name: target, Length: 1025, dtype: int64

In [11]:
x

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal
0,52,1,0,125,212,0,1,168,0,1.0,2,2,3
1,53,1,0,140,203,1,0,155,1,3.1,0,0,3
2,70,1,0,145,174,0,1,125,1,2.6,0,0,3
3,61,1,0,148,203,0,1,161,0,0.0,2,1,3
4,62,0,0,138,294,1,1,106,0,1.9,1,3,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1020,59,1,1,140,221,0,1,164,1,0.0,2,0,2
1021,60,1,0,125,258,0,0,141,1,2.8,1,1,3
1022,47,1,0,110,275,0,0,118,1,1.0,1,1,2
1023,50,0,0,110,254,0,0,159,0,0.0,2,0,2


### Train Test split , Data scalling

In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42, stratify=y
)

In [14]:
test_df = X_test.copy()
test_df["target"] = y_test
test_df.to_csv("test_data.csv", index=False)
print("Saved test_data.csv successfully.")

Saved test_data.csv successfully.


In [15]:
os.makedirs("models", exist_ok=True)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
joblib.dump(scaler, "models/scaler.pkl")

['models/scaler.pkl']

## Model Building

### Base Model

In [18]:
models = {
    "Logistic_Regression": LogisticRegression(random_state=42),
    "Decision_Tree": DecisionTreeClassifier(random_state=42, max_depth=5),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "Naive_Bayes": GaussianNB(),
    "SVM": SVC(probability=True, random_state=42)
}

In [19]:
results = []
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    joblib.dump(model, f"models/{name}.pkl")

    # Predict
    y_pred = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)[:, 1]

    # Metrics
    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    mcc = matthews_corrcoef(y_test, y_pred)

    results.append({
        "Model": name,
        "Accuracy": round(acc, 4),
        "AUC": round(auc, 4),
        "Precision": round(prec, 4),
        "Recall": round(rec, 4),
        "F1": round(f1, 4),
        "MCC": round(mcc, 4)
    })

/home/cloud/anaconda3/lib/python3.12/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


In [20]:
results_df = pd.DataFrame(results)
print("\nModel Evaluation Comparison Table:")
print(results_df.to_string(index=False))


Model Evaluation Comparison Table:
              Model  Accuracy    AUC  Precision  Recall     F1    MCC
Logistic_Regression    0.8098 0.9298     0.7619  0.9143 0.8312 0.6309
      Decision_Tree    0.8732 0.9326     0.8624  0.8952 0.8785 0.7465
                KNN    0.8634 0.9629     0.8738  0.8571 0.8654 0.7269
        Naive_Bayes    0.8293 0.9043     0.8070  0.8762 0.8402 0.6602
                SVM    0.9268 0.9771     0.9167  0.9429 0.9296 0.8538


### Model Finetuning

In [22]:
# Model Definitions & Hyperparameter Grids

model_pipeline_config = {
    "Logistic_Regression": {
        "model": LogisticRegression(random_state=42),
        "params": {
            "C": [0.01, 0.1, 1.0, 10.0],
            "solver": ["lbfgs", "liblinear"],
            "max_iter": [200, 500]
        }
    },
    "Decision_Tree": {
        "model": DecisionTreeClassifier(random_state=42),
        "params": {
            "max_depth": [3, 5, 7, 10],
            "min_samples_split": [2, 5, 10],
            "criterion": ["gini", "entropy"]
        }
    },
    "KNN": {
        "model": KNeighborsClassifier(),
        "params": {
            "n_neighbors": [3, 5, 7, 9, 11],
            "weights": ["uniform", "distance"],
            "metric": ["euclidean", "manhattan"]
        }
    },
    "Naive_Bayes": {
        "model": GaussianNB(),
        "params": {
            "var_smoothing": np.logspace(0, -9, num=10)
        }
    },
    "SVM": {
        "model": SVC(probability=True, random_state=42),
        "params": {
            "C": [0.01,0.1, 1, 10],
            "kernel": ["rbf", "linear"],
            "gamma": ["scale", "auto"]
        }
    }
}

# Grid Search Tuning, Evaluation, & Export

results = []
print("Starting Hyperparameter Tuning with GridSearchCV...\n")

for name, config in model_pipeline_config.items():
    print(f"Tuning {name}...")

    # Run 5-Fold Cross-Validation Grid Search
    grid_search = GridSearchCV(
        estimator=config["model"],
        param_grid=config["params"],
        cv=5,
        scoring="f1",  # Optimize for balanced precision/recall
        n_jobs=-1
    )
    grid_search.fit(X_train_scaled, y_train)

    # Retrieve best model estimator
    best_model = grid_search.best_estimator_
    joblib.dump(best_model, f"models/{name}.pkl")

    # Evaluate on held-out test data
    y_pred = best_model.predict(X_test_scaled)
    if hasattr(best_model, "predict_proba"):
        y_proba = best_model.predict_proba(X_test_scaled)[:, 1]
    else:
        y_proba = best_model.decision_function(X_test_scaled)

    # Compute assignment metrics
    results.append({
        "Model": name,
        "Best Params": str(grid_search.best_params_),
        "Accuracy": round(accuracy_score(y_test, y_pred), 4),
        "AUC": round(roc_auc_score(y_test, y_proba), 4),
        "Precision": round(precision_score(y_test, y_pred), 4),
        "Recall": round(recall_score(y_test, y_pred), 4),
        "F1": round(f1_score(y_test, y_pred), 4),
        "MCC": round(matthews_corrcoef(y_test, y_pred), 4)
    })

# Display tuned model metrics
results_df = pd.DataFrame(results)
print("\nTuned Models Performance Summary:")
print(results_df.drop(columns=["Best Params"]).to_string(index=False))

print("\nOptimal Hyperparameters:")
for idx, row in results_df.iterrows():
    print(f"• {row['Model']}: {row['Best Params']}")

Starting Hyperparameter Tuning with GridSearchCV...

Tuning Logistic_Regression...
Tuning Decision_Tree...
Tuning KNN...
Tuning Naive_Bayes...
Tuning SVM...


/home/cloud/anaconda3/lib/python3.12/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/cloud/anaconda3/lib/python3.12/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/cloud/anaconda3/lib/python3.12/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/cloud/anaconda3/lib/python3.12/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will 


Tuned Models Performance Summary:
              Model  Accuracy    AUC  Precision  Recall     F1    MCC
Logistic_Regression    0.8098 0.9298     0.7619  0.9143 0.8312 0.6309
      Decision_Tree    1.0000 1.0000     1.0000  1.0000 1.0000 1.0000
                KNN    1.0000 1.0000     1.0000  1.0000 1.0000 1.0000
        Naive_Bayes    0.8244 0.9105     0.7634  0.9524 0.8475 0.6686
                SVM    0.9854 1.0000     0.9722  1.0000 0.9859 0.9711

Optimal Hyperparameters:
• Logistic_Regression: {'C': 1.0, 'max_iter': 200, 'solver': 'lbfgs'}
• Decision_Tree: {'criterion': 'entropy', 'max_depth': 10, 'min_samples_split': 2}
• KNN: {'metric': 'euclidean', 'n_neighbors': 9, 'weights': 'distance'}
• Naive_Bayes: {'var_smoothing': np.float64(1.0)}
• SVM: {'C': 10, 'gamma': 'auto', 'kernel': 'rbf'}


/home/cloud/anaconda3/lib/python3.12/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


### Training the model with optimized hyperparameters

In [24]:
fine_tuned_models = {
    "Logistic_Regression": LogisticRegression(random_state=42, C=1, max_iter=200, solver='lbfgs'),
    "Decision_Tree": DecisionTreeClassifier(random_state=42, max_depth=10, criterion='entropy', min_samples_split=2),
    "KNN": KNeighborsClassifier(n_neighbors=9, weights='distance', metric='euclidean'),
    "Naive_Bayes": GaussianNB(var_smoothing = np.float64(1.0)),
    "SVM": SVC(probability=True, random_state=42, C=10, gamma='auto', kernel='rbf')
}

In [25]:
fine_tuned_results = []
for name, model in fine_tuned_models.items():
    model.fit(X_train_scaled, y_train)
    joblib.dump(model, f"models/{name}.pkl")

    # Predict
    y_pred = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)[:, 1]

    # Metrics
    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    mcc = matthews_corrcoef(y_test, y_pred)

    fine_tuned_results.append({
        "Model": name,
        "Accuracy": round(acc, 4),
        "AUC": round(auc, 4),
        "Precision": round(prec, 4),
        "Recall": round(rec, 4),
        "F1": round(f1, 4),
        "MCC": round(mcc, 4)
    })

/home/cloud/anaconda3/lib/python3.12/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


In [26]:
results_df = pd.DataFrame(fine_tuned_results)
print("\nFine Tuned Model Evaluation Comparison Table:")
print(results_df.to_string(index=False))


Fine Tuned Model Evaluation Comparison Table:
              Model  Accuracy    AUC  Precision  Recall     F1    MCC
Logistic_Regression    0.8098 0.9298     0.7619  0.9143 0.8312 0.6309
      Decision_Tree    1.0000 1.0000     1.0000  1.0000 1.0000 1.0000
                KNN    1.0000 1.0000     1.0000  1.0000 1.0000 1.0000
        Naive_Bayes    0.8244 0.9105     0.7634  0.9524 0.8475 0.6686
                SVM    0.9854 1.0000     0.9722  1.0000 0.9859 0.9711
